# training/validation visualization

this notebook visualizes tile-level predictions over the training and validation split using the current repository api.


In [ ]:
# === parameters ===
MODEL_PATH = "models/best_model_f1.pth"
MAX_DEPTH_BLOCKS = None  # set an int to limit depth blocks for quick tests


In [ ]:
# === imports and setup ===
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from utils.config import Config
from utils.dataloader import DataManager
from utils.model import create_model
from utils.visualizer import group_by_depth, predict_tiles

config = Config()
device = config.device
print(f"device: {device}; tra_scroll_id: {config.data.tra_scroll_id}")


In [ ]:
# === load model ===
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"model file not found: {MODEL_PATH}")

print("loading model...")
model, _ = create_model(config)
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.to(device)
model.eval()
print("model loaded")


In [ ]:
# === load train/valid data from current dataloader api ===
dm = DataManager(config)
volume = dm.vol
mask = dm.mask
labels = dm.labels
train_x_range = dm.train_x
valid_x_range = dm.valid_x
y_range = dm.y_range
global_mean, global_std, global_min, global_max = dm.norm_stats

print(f"volume shape: {volume.shape}")
print(f"train x_range: {train_x_range}, valid x_range: {valid_x_range}")
print(f"y_range: {y_range}")
print(f"norm stats: mean={global_mean:.4f} std={global_std:.4f} min={global_min:.4f} max={global_max:.4f}")


In [ ]:
# === tile coordinate generation matching trainer/visualizer logic ===
def gen_tile_coords(config, z_range, y_range, x_range, mask):
    z0, z1 = z_range
    y0, y1 = y_range
    x0, x1 = x_range

    depth = config.data.depth
    tile = config.data.tile_size

    z_span = max(0, z1 - z0 - depth + 1)
    y_span = max(0, y1 - y0 - tile + 1)
    x_span = max(0, x1 - x0 - tile + 1)

    coords = []
    z_step = max(1, depth // 2)

    for d_off in range(0, z_span, z_step):
        if z0 + d_off + depth > z1:
            continue
        for y_off in range(0, y_span, tile):
            for x_off in range(0, x_span, tile):
                m_tile = mask[y0 + y_off:y0 + y_off + tile, x0 + x_off:x0 + x_off + tile]
                if np.sum(m_tile) > 0:
                    coords.append((d_off, y_off, x_off))

    return coords

z_range = (config.data.d_start, config.data.d_end)
train_coords = gen_tile_coords(config, z_range, y_range, train_x_range, mask)
valid_coords = gen_tile_coords(config, z_range, y_range, valid_x_range, mask)

train_grouped = group_by_depth(train_coords)
valid_grouped = group_by_depth(valid_coords)
all_depth_offsets = sorted(set(train_grouped.keys()) | set(valid_grouped.keys()))

if MAX_DEPTH_BLOCKS is not None:
    all_depth_offsets = all_depth_offsets[:MAX_DEPTH_BLOCKS]

print(f"train tiles: {len(train_coords)}")
print(f"valid tiles: {len(valid_coords)}")
print(f"depth blocks: {len(all_depth_offsets)}")


In [ ]:
# === run prediction for each depth block and build figures ===
all_pred_data = []

for d_off in all_depth_offsets:
    depth_start = d_off + config.data.d_start
    depth_end = depth_start + config.data.depth

    t_coords = train_grouped.get(d_off, [])
    v_coords = valid_grouped.get(d_off, [])

    print(f"predicting depth block {depth_start}-{depth_end} (train: {len(t_coords)}, valid: {len(v_coords)})")

    train_pred = predict_tiles(
        config,
        model,
        volume,
        mask,
        t_coords,
        y_range,
        train_x_range,
        depth_start,
        "train",
        global_mean,
        global_std,
        global_min,
        global_max,
    )

    valid_pred = predict_tiles(
        config,
        model,
        volume,
        mask,
        v_coords,
        y_range,
        valid_x_range,
        depth_start,
        "valid",
        global_mean,
        global_std,
        global_min,
        global_max,
    )

    full_pred = np.concatenate([train_pred, valid_pred], axis=1)
    all_pred_data.append((full_pred, train_pred.shape[1], depth_start, depth_end))

print(f"generated predictions for {len(all_pred_data)} depth blocks")


In [ ]:
# === visualize with train/valid split marker and label overlay ===
tile = config.data.tile_size
labels_crop = labels[y_range[0]:y_range[1], train_x_range[0]:valid_x_range[1]]
labels_small = labels_crop[::tile, ::tile]

for full_pred, train_width, depth_start, depth_end in all_pred_data:
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    ax_pred = axes[0]
    ax_pred.imshow(full_pred, cmap="inferno", vmin=0, vmax=1)
    ax_pred.axvline(x=train_width - 0.5, color="red", linestyle="--", linewidth=1.2)
    ax_pred.set_title(f"predictions depth {depth_start}-{depth_end}")
    ax_pred.axis("off")

    ax_overlay = axes[1]
    ax_overlay.imshow(full_pred, cmap="inferno", vmin=0, vmax=1)

    overlay = np.zeros((*full_pred.shape, 4), dtype=np.float32)
    h = min(labels_small.shape[0], overlay.shape[0])
    w = min(labels_small.shape[1], overlay.shape[1])
    overlay[:h, :w][labels_small[:h, :w] > 0.5] = [1, 1, 1, 0.35]

    ax_overlay.imshow(overlay)
    ax_overlay.axvline(x=train_width - 0.5, color="red", linestyle="--", linewidth=1.2)
    ax_overlay.set_title(f"overlay depth {depth_start}-{depth_end}")
    ax_overlay.axis("off")

    plt.tight_layout()
    plt.show()
